In [1]:
import pandas as pd
import os
import sys
import plotly.express as px
import warnings

# Ignorar avisos irrelevantes do pandas
warnings.filterwarnings('ignore')

# Define a pasta local onde estão os seus CSVs
pasta_dados = 'data/'
dfs = []

print("Iniciando a leitura dos datasets locais...")

# Lista todos os arquivos terminados em .csv dentro da pasta 'data'
arquivos_csv = [f for f in os.listdir(pasta_dados) if f.endswith('.csv')]

for arquivo in arquivos_csv:
    caminho_completo = os.path.join(pasta_dados, arquivo)
    print(f"Carregando {arquivo}...")
    try:
        # Lê o arquivo direto do seu HD (D:) usando as mesmas regras de formatação
        df = pd.read_csv(caminho_completo, sep=';', encoding='latin1', on_bad_lines='skip', low_memory=False)
        dfs.append(df)
    except Exception as e:
        print(f" -> Erro ao ler o arquivo {arquivo}: {e}")

# Verificação de segurança (Parada se a pasta estiver vazia ou com erro)
if not dfs:
    print("ERRO CRÍTICO: Nenhum dataset foi carregado. Verifique se os arquivos estão na pasta 'data'. Execução interrompida.")
    sys.exit()

# Concatenação de todos os anos
df_all = pd.concat(dfs, ignore_index=True)
print(f"\nLeitura concluída com sucesso! Volume total da base: {len(df_all)} registros.")

Iniciando a leitura dos datasets locais...
Carregando emlurb_2025.csv...
Carregando resources_031c3ad3-265f-4d9c-830a-7d1f85f830fa_2024-central-de-atendimento-de-servicos-da-emlurb-156.csv...
Carregando resources_192dc99d-b248-4671-8b3d-a8fe7210bf3c_2021-central-de-atendimento-de-servicos-da-emlurb-156.csv...
Carregando resources_30ba6cc0-17a8-4f38-8c86-55184c87ead1_2023-central-de-atendimento-de-servicos-da-emlurb-156.csv...
Carregando resources_6c7fceed-3495-449c-bbd6-01bcd3e760bd_2022-central-de-atendimento-de-servicos-da-emlurb-156.csv...
Carregando resources_aba3d1a0-c456-4a3b-91e1-4aaf30704717_2020-central-de-atendimento-de-servicos-da-emlurb-156.csv...

Leitura concluída com sucesso! Volume total da base: 572826 registros.


In [2]:
print("Iniciando rotina de limpeza e engenharia de dados...")

# 1. Tratamento de Datas
df_all['DATA_DEMANDA'] = pd.to_datetime(df_all['DATA_DEMANDA'], format='mixed', errors='coerce')
df_all['DATA_ULT_SITUACAO'] = pd.to_datetime(df_all['DATA_ULT_SITUACAO'], format='mixed', errors='coerce')

# 2. Remoção Estrita de Nulos
reg_antes = len(df_all)
df_all.dropna(subset=['DATA_DEMANDA'], inplace=True)
print(f"Linhas descartadas por falta de DATA_DEMANDA: {reg_antes - len(df_all)}")

# 3. Engenharia de Variáveis (Features) para EDA
df_all['Ano_Mes'] = df_all['DATA_DEMANDA'].dt.to_period('M').astype(str)
df_all['Mes'] = df_all['DATA_DEMANDA'].dt.month
# Extraindo o nome do dia da semana
df_all['Dia_Semana'] = df_all['DATA_DEMANDA'].dt.day_name()

# 4. Padronização de Strings (Tirar espaços invisíveis e deixar tudo maiúsculo)
df_all['BAIRRO'] = df_all['BAIRRO'].astype(str).str.strip().str.upper()
df_all['GRUPOSERVICO_DESCRICAO'] = df_all['GRUPOSERVICO_DESCRICAO'].astype(str).str.strip().str.upper()

print("Limpeza finalizada. A base está pronta para a Análise Exploratória.")

Iniciando rotina de limpeza e engenharia de dados...
Linhas descartadas por falta de DATA_DEMANDA: 0
Limpeza finalizada. A base está pronta para a Análise Exploratória.


In [3]:
vol_mensal = df_all['Ano_Mes'].value_counts().sort_index().reset_index()
vol_mensal.columns = ['Mês', 'Volume de Denúncias']

fig1 = px.line(vol_mensal, x='Mês', y='Volume de Denúncias', 
               title='1. Evolução do Volume Total de Denúncias (Série Histórica)',
               markers=True)
fig1.update_layout(xaxis_tickangle=-45)
fig1.show()

In [4]:
top5_cat = df_all['GRUPOSERVICO_DESCRICAO'].value_counts().head(5).index
df_top5_cat = df_all[df_all['GRUPOSERVICO_DESCRICAO'].isin(top5_cat)]

vol_mes_cat = df_top5_cat.groupby(['Mes', 'GRUPOSERVICO_DESCRICAO']).size().reset_index(name='Volume')

fig2 = px.bar(vol_mes_cat, x='Mes', y='Volume', color='GRUPOSERVICO_DESCRICAO',
              barmode='group', title='2. Sazonalidade das Categorias (Top 5 Serviços por Mês)',
              labels={'Mes': 'Mês do Ano (1 a 12)', 'GRUPOSERVICO_DESCRICAO': 'Categoria'})
fig2.show()